In [ ]:
# Created on Oct 12, 2025
# Created by: Afeena
# Purpose: This file answers the business questions for the user slice

import pandas as pd
import numpy as np
import os as os
import statsmodels.formula.api as smf

# File Paths
UserSeveritySlicePath = "C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Master Code (BA)\\04Regression\\UserSeveritySlice.csv"
destinationFolder = "C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Master Code (BA)\\04Regression"

# Read the CSV file
dfUser = pd.read_csv(UserSeveritySlicePath, low_memory=False)

# Make column for SevereInjury:Fatal/Major = 1, Minor/Minimal = 0
dfUser['SevereInjury'] = np.where(dfUser['SeverityOfInjury'].isin(['Fatal', 'Major']), 1, 0)

# print(df['SeverityOfInjury'].value_counts())
# print(df['SevereInjury'].value_counts())

# identify dummy variable columns
userInvTypeCols = ["UserInvType_Cyclist",
"UserInvType_Cyclist_Passenger",
"UserInvType_Driver___Not_Hit",
"UserInvType_In_Line_Skater",
"UserInvType_Moped_Driver",
"UserInvType_Moped_Passenger",
"UserInvType_Motorcycle_Driver",
"UserInvType_Motorcycle_Passenger",
"UserInvType_Other",
"UserInvType_Passenger",
"UserInvType_Pedestrian",
"UserInvType_Truck_Driver",
"UserInvType_Wheelchair"
]
userAgeRangeCols = ["UserAgeRange_0_to_4",
"UserAgeRange_10_to_14",
"UserAgeRange_15_to_19",
"UserAgeRange_25_to_29",
"UserAgeRange_30_to_34",
"UserAgeRange_35_to_39",
"UserAgeRange_40_to_44",
"UserAgeRange_45_to_49",
"UserAgeRange_5_to_9",
"UserAgeRange_50_to_54",
"UserAgeRange_55_to_59",
"UserAgeRange_60_to_64",
"UserAgeRange_65_to_69",
"UserAgeRange_70_to_74",
"UserAgeRange_75_to_79",
"UserAgeRange_80_to_84",
"UserAgeRange_85_to_89",
"UserAgeRange_90_to_94",
"UserAgeRange_Over_95",
"UserAgeRange_unknown"
]

# ModelA: Severity- Involvement Types
formulaA = 'SevereInjury ~ ' + ' + '.join(userInvTypeCols)
modelA = smf.logit(formula=formulaA, data=dfUser).fit()
print("\n Model A: User Involvement Types vs Severe Injuries \n")
print(modelA.summary())

# Save model summary to csv file
with open(os.path.join(destinationFolder, "UserModelA_Stats.csv"), 'w') as f:
    f.write(str(modelA.summary()))

# ModelB: Severity- User Age Ranges
formulaB = 'SevereInjury ~ ' + ' + '.join(userAgeRangeCols)
modelB = smf.logit(formula=formulaB, data=dfUser).fit()
print("\n Model B: User Age Ranges vs Severe Injuries \n")
print(modelB.summary())

# Save model summary to csv file
with open(os.path.join(destinationFolder, "UserModelB_Stats.csv"), 'w') as f:
    f.write(str(modelB.summary()))

# Function for Odds Ratios and Confidence Intervals
# the Odd Ratios and Summary function above will give the same results.
def odds_ratios_and_ci(whichModel):
    params = whichModel.params
    conf = whichModel.conf_int()
    out = pd.DataFrame({
        # Odds Ratios
        "OR": np.exp(params),
        # Confidence Intervals
        "2.5%": np.exp(conf[0]),
        "97.5%": np.exp(conf[1]),
        # p-values
        "pval": whichModel.pvalues
    })
    return out.sort_values(by="OR", ascending=False)

# Predicted probability for each involvement dummy (set one=1, others=0; all other dummies=0)
def group_pred_by_singleton(whichModel, whichColumns, group_name="Group"):
    # create baseline scenario - set all dummy variables to 0 (reference categories). I will turn on what I want to test
    base = {col: 0 for col in whichModel.model.exog_names if col not in ["Intercept"]}
    results = []     # list to store prediction results
    for g in whichColumns:
        scenario = base.copy() # creates a copy
        scenario[g] = 1  # Set only the current group dummy to 1, others stay 0, allowing me to loop through each dummy variable set it to 1 and then leave the others at 0
        prob = whichModel.predict(pd.DataFrame([scenario]))[0]
        results.append({group_name: g, "PredProb_Severe": prob})
    return pd.DataFrame(results).sort_values("PredProb_Severe", ascending=False)

#Print and save Predicted probabilities for Models A and Model B
print("\n=== Predicted probability of severe injury by involvement type ===")
predProb_involvement = group_pred_by_singleton(modelA, userInvTypeCols, group_name="Involvement")
display(predProb_involvement)
predProb_involvement.to_csv(os.path.join(destinationFolder, "UserModelA_Stats.csv"),  mode='a', header=True,index=False)

print("\n=== Predicted probability of severe injury by age range ===")
predProb_age = group_pred_by_singleton(modelB, userAgeRangeCols, group_name="AgeRange")
display(predProb_age)
predProb_age.to_csv(os.path.join(destinationFolder, "UserModelB_Stats.csv"), mode='a', header=True, index=False)

# Print and save outputs of odds ratios
modelA_or_ci = odds_ratios_and_ci(modelA)
print("\n=== Odds Ratios for Model A (involvement type) ===")
display(modelA_or_ci)
modelA_or_ci.to_csv(os.path.join(destinationFolder, "UserModelA_Stats.csv"), mode='a', header=True, index=False)

modelB_or_ci = odds_ratios_and_ci(modelB)
print("\n=== Odds Ratios for Model B (Age Range) ===")
display(modelB_or_ci)
modelB_or_ci.to_csv(os.path.join(destinationFolder, "UserModelB_Stats.csv"), mode='a', header=True, index=False)


involvement dummies: ['UserInvType_Cyclist', 'UserInvType_Cyclist_Passenger', 'UserInvType_Driver___Not_Hit', 'UserInvType_In_Line_Skater', 'UserInvType_Moped_Driver'] ...total: 13
age range dummies: ['UserAgeRange_0_to_4', 'UserAgeRange_10_to_14', 'UserAgeRange_15_to_19', 'UserAgeRange_25_to_29', 'UserAgeRange_30_to_34'] ...total: 20
         Current function value: 0.440483
         Iterations: 35

 Model A: User Involvement Types vs Severe Injuries 

                           Logit Regression Results                           
Dep. Variable:           SevereInjury   No. Observations:                10044
Model:                          Logit   Df Residuals:                    10030
Method:                           MLE   Df Model:                           13
Date:                Sun, 12 Oct 2025   Pseudo R-squ.:                  0.2347
Time:                        12:59:53   Log-Likelihood:                -4424.2
converged:                      False   LL-Null:                    

c:\Users\afeen\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\afeen\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


         Current function value: 0.564000
         Iterations: 35

 Model B: User Age Ranges vs Severe Injuries 

                           Logit Regression Results                           
Dep. Variable:           SevereInjury   No. Observations:                10044
Model:                          Logit   Df Residuals:                    10023
Method:                           MLE   Df Model:                           20
Date:                Sun, 12 Oct 2025   Pseudo R-squ.:                 0.02015
Time:                        12:59:53   Log-Likelihood:                -5664.8
converged:                      False   LL-Null:                       -5781.3
Covariance Type:            nonrobust   LLR p-value:                 2.994e-38
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.7955      0.063     12.563      0.000       0.

,Involvement,PredProb_Severe
12,UserInvType_Wheelchair,1.000000e+00
3,UserInvType_In_Line_Skater,1.000000e+00
5,UserInvType_Moped_Passenger,1.000000e+00
0,UserInvType_Cyclist,9.788557e-01
6,UserInvType_Motorcycle_Driver,9.692737e-01
4,UserInvType_Moped_Driver,9.655172e-01
10,UserInvType_Pedestrian,9.487496e-01
7,UserInvType_Motorcycle_Passenger,7.000000e-01
1,UserInvType_Cyclist_Passenger,6.666667e-01
9,UserInvType_Passenger,4.725738e-01



=== Predicted probability of severe injury by age range ===


...


=== Odds Ratios for Model A (involvement type) ===


c:\Users\afeen\anaconda3\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: overflow encountered in exp
  result = getattr(ufunc, method)(*inputs, **kwargs)


,OR,2.5%,97.5%,pval
UserInvType_Wheelchair,1.268277e+09,0.000000,inf,9.983307e-01
UserInvType_In_Line_Skater,1.234460e+09,0.000000,inf,9.991803e-01
UserInvType_Moped_Passenger,2.062256e+08,0.000000,inf,9.990830e-01
UserInvType_Cyclist,3.440779e+01,21.173987,55.912758,2.762442e-46
UserInvType_Motorcycle_Driver,2.344595e+01,15.249722,36.047371,7.456000e-47
UserInvType_Moped_Driver,2.081081e+01,2.828074,153.139514,2.874459e-03
UserInvType_Pedestrian,1.375895e+01,11.595644,16.325858,3.073746e-198
UserInvType_Motorcycle_Passenger,1.734234e+00,0.878730,3.422630,1.124533e-01
UserInvType_Cyclist_Passenger,1.486486e+00,0.134653,16.409909,7.462901e-01
Intercept,1.345455e+00,1.254760,1.442704,7.837032e-17



=== Odds Ratios for Model B (Age Range) ===


,OR,2.5%,97.5%,pval
UserAgeRange_Over_95,196.777112,0.000227,1.706203e+08,4.489491e-01
UserAgeRange_90_to_94,11.735660,2.843073,4.844255e+01,6.629349e-04
UserAgeRange_85_to_89,4.086100,2.503719,6.668567e+00,1.777197e-08
UserAgeRange_80_to_84,2.757026,1.906119,3.987785e+00,7.223816e-08
Intercept,2.215470,1.956899,2.508206e+00,3.382374e-36
UserAgeRange_75_to_79,2.186817,1.605978,2.977730e+00,6.778581e-07
UserAgeRange_70_to_74,2.015279,1.514457,2.681719e+00,1.529532e-06
UserAgeRange_65_to_69,1.780955,1.372841,2.310392e+00,1.384712e-05
UserAgeRange_60_to_64,1.678282,1.314427,2.142856e+00,3.284612e-05
UserAgeRange_55_to_59,1.384763,1.115269,1.719379e+00,3.199394e-03
